# SQL Exercises: Payments Data Analysis

A hands-on SQL course using the `payments.default` catalog. Each cell builds on the last — run them top to bottom.

**Tables**

| Table | Key Columns | Description |
|---|---|---|
| `pay_users` | `id`, `current_age`, `gender`, `yearly_income`, `credit_score`, `total_debt` | One row per client |
| `pay_cards` | `id`, `client_id`, `card_brand`, `card_type`, `credit_limit`, `has_chip` | One row per card |
| `pay_transactions` | `id`, `client_id`, `card_id`, `amount`, `use_chip`, `is_online`, `merchant_state` | One row per transaction |
| `pay_labels` | `transaction_id`, `is_fraud` | Fraud label per transaction |

**Join keys**
* `pay_transactions.client_id` → `pay_users.id`
* `pay_transactions.card_id` → `pay_cards.id`
* `pay_labels.transaction_id` → `pay_transactions.id`

**Note:** Dollar columns (`yearly_income`, `credit_limit`, `total_debt`, `per_capita_income`) are stored as strings prefixed with `$`.

In [0]:
%sql
-- ============================================================
-- Exercise 1: SELECT & LIMIT — Preview the Data
-- ============================================================
-- Preview 5 rows from the transactions table to understand columns.

SELECT id, date, client_id, card_id, use_chip,
       merchant_city, merchant_state, is_online, amount
FROM payments.default.pay_transactions
LIMIT 5;

In [0]:
%sql
-- ============================================================
-- Exercise 2: WHERE & ORDER BY — Top 10 Largest Online Transactions
-- ============================================================
-- Filter for online transactions, sort by amount descending.

SELECT id, date, client_id, amount,
       merchant_city, merchant_state
FROM payments.default.pay_transactions
WHERE is_online = 'ONLINE'
ORDER BY amount DESC
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Exercise 3: Aggregations & GROUP BY — Stats by Card Brand
-- ============================================================
-- Join transactions → cards, then compute count, avg, and total
-- transaction amount per card brand.

SELECT c.card_brand,
       COUNT(*)                  AS num_transactions,
       ROUND(AVG(t.amount), 2)   AS avg_amount,
       ROUND(SUM(t.amount), 2)   AS total_amount
FROM payments.default.pay_transactions t
JOIN payments.default.pay_cards c ON t.card_id = c.id
GROUP BY c.card_brand
ORDER BY total_amount DESC;

In [0]:
%sql
-- ============================================================
-- Exercise 4: INNER JOIN — Transactions with User Demographics
-- ============================================================
-- Join transactions → users to see who made each high-value
-- transaction (> $500).

SELECT t.id AS transaction_id,
       t.amount,
       t.date,
       u.current_age,
       u.gender,
       u.credit_score
FROM payments.default.pay_transactions t
JOIN payments.default.pay_users u ON t.client_id = u.id
WHERE t.amount > 500
ORDER BY t.amount DESC
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Exercise 5: GROUP BY + HAVING — Busy Merchant States
-- ============================================================
-- Find states with more than 10,000 transactions and their
-- average transaction amount.

SELECT merchant_state,
       COUNT(*)              AS num_transactions,
       ROUND(AVG(amount), 2) AS avg_amount
FROM payments.default.pay_transactions
WHERE merchant_state IS NOT NULL
GROUP BY merchant_state
HAVING COUNT(*) > 10000
ORDER BY num_transactions DESC;

In [0]:
%sql
-- ============================================================
-- Exercise 6: Subquery — Users with Above-Average Credit Score
-- ============================================================
-- Find users whose credit score exceeds the population average.

SELECT id, current_age, gender, credit_score, num_credit_cards
FROM payments.default.pay_users
WHERE credit_score > (SELECT AVG(credit_score)
                      FROM payments.default.pay_users)
ORDER BY credit_score DESC
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Exercise 7: CASE & String Functions — Clean & Categorize
-- ============================================================
-- Strip '$' from dollar columns, compute a debt-to-income ratio,
-- and categorize users into credit bands with CASE.

SELECT id,
       current_age,
       gender,
       credit_score,
       CAST(REPLACE(yearly_income, '$', '') AS DOUBLE) AS income_num,
       CAST(REPLACE(total_debt, '$', '')    AS DOUBLE) AS debt_num,
       ROUND(CAST(REPLACE(total_debt, '$', '') AS DOUBLE) /
             NULLIF(CAST(REPLACE(yearly_income, '$', '') AS DOUBLE), 0), 2) AS debt_to_income,
       CASE
         WHEN credit_score >= 750 THEN 'Excellent'
         WHEN credit_score >= 700 THEN 'Good'
         WHEN credit_score >= 650 THEN 'Fair'
         ELSE 'Poor'
       END AS credit_band
FROM payments.default.pay_users
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Exercise 8: Window Functions — Largest Transaction per User
-- ============================================================
-- Use ROW_NUMBER() to rank each user's transactions by amount,
-- then keep only their single largest.

WITH ranked AS (
  SELECT id,
         client_id,
         amount,
         date,
         ROW_NUMBER() OVER (PARTITION BY client_id
                            ORDER BY amount DESC) AS rn
  FROM payments.default.pay_transactions
)
SELECT client_id, id AS transaction_id, amount, date
FROM ranked
WHERE rn = 1
ORDER BY amount DESC
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- Exercise 9: CTE — Fraud Summary by Card Brand
-- ============================================================
-- Join transactions → labels → cards, then compute fraud count
-- and fraud rate (%) per card brand using conditional aggregation.

WITH fraud_counts AS (
  SELECT c.card_brand,
         l.is_fraud,
         COUNT(*) AS cnt
  FROM payments.default.pay_transactions t
  JOIN payments.default.pay_labels l ON t.id = l.transaction_id
  JOIN payments.default.pay_cards  c ON t.card_id = c.id
  GROUP BY c.card_brand, l.is_fraud
)
SELECT card_brand,
       SUM(CASE WHEN is_fraud = 'Yes' THEN cnt ELSE 0 END) AS fraud_count,
       SUM(CASE WHEN is_fraud = 'No'  THEN cnt ELSE 0 END) AS legit_count,
       SUM(cnt) AS total_count,
       ROUND(SUM(CASE WHEN is_fraud = 'Yes' THEN cnt ELSE 0 END) * 100.0 /
             NULLIF(SUM(cnt), 0), 2) AS fraud_rate_pct
FROM fraud_counts
GROUP BY card_brand
ORDER BY fraud_rate_pct DESC;

In [0]:
%sql
-- ============================================================
-- Exercise 10: Advanced — Fraud Rate by Online Status & Card Type
-- ============================================================
-- Combine all four tables to analyze fraud rates across card type
-- and online/offline transactions, including avg credit score.

WITH tx_enriched AS (
  SELECT t.id,
         t.is_online,
         t.amount,
         c.card_brand,
         c.card_type,
         l.is_fraud,
         u.credit_score
  FROM payments.default.pay_transactions t
  JOIN payments.default.pay_cards  c ON t.card_id   = c.id
  JOIN payments.default.pay_users  u ON t.client_id = u.id
  JOIN payments.default.pay_labels l ON t.id        = l.transaction_id
)
SELECT card_brand,
       card_type,
       is_online,
       COUNT(*)                    AS num_transactions,
       ROUND(AVG(amount), 2)       AS avg_amount,
       ROUND(AVG(credit_score), 0) AS avg_credit_score,
       SUM(CASE WHEN is_fraud = 'Yes' THEN 1 ELSE 0 END) AS fraud_count,
       ROUND(SUM(CASE WHEN is_fraud = 'Yes' THEN 1 ELSE 0 END) * 100.0 /
             NULLIF(COUNT(*), 0), 2) AS fraud_rate_pct
FROM tx_enriched
GROUP BY card_brand, card_type, is_online
ORDER BY fraud_rate_pct DESC;